# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #3 — "Click Capture by Position Tier"** (`docs/flyrank-seo-research-march-2026.pdf`).
The paper reports weighted portfolio CTR falling from 0.423% (Top-3) to 0.050% (Deep) — an 88%
drop — computed as total clicks ÷ total impressions per position tier, across 341,701 pages /
57 brands.

*Where does the label come from?* It isn't really a predictive label at all — it's a direct
aggregate ratio (clicks/impressions), same as ML-04/ML-06's `ctr_gap`. That's a strength, not a
weakness: there's no proxy-label ambiguity to interrogate here.
*Does the validation design carry the claim?* The paper is explicit that these are descriptive,
portfolio-level comparisons, not a trained/tested model — so "validation design" isn't really
the right lens for this one. My own Signal #1 (ML-06) measured the same direction independently,
on a different data slice, with the same conclusion. Two independent measurements agreeing is
about as much confidence as this kind of comparison can honestly offer.

**Finding #4 — "The Freshness Multiplier"**. The paper tags this **CONFIRMED** and calls refresh
timing "one of the strongest measured levers available," citing a 31-90 day growth-to-decline
ratio of 7.88:1, and a 361+ day bucket spiking to 283:1.

*Where does the label come from?* "Growth-to-decline ratio" is built from `trend_direction`,
itself derived from a 30d-vs-prev-30d impression comparison (per the paper's own metric
glossary) — a different construction than my `ctr_gap`/decline check, but a comparable idea:
compare an earlier period to a later one.
*Does the validation design carry the claim?* Here I'd push back, constructively: the paper's
own text admits the 361+ bucket's 283:1 figure comes from **a single declining page** in that
bucket — that's not a "measured lever," that's one data point dressed up as a ratio. More
importantly, **my own Signal #2 (ML-06) tested essentially the same idea — staleness vs.
decline — on this warehouse slice, and got FALSE**: `pct_declined` sat flat at 52-54% across
every staleness bucket. The paper and my own data disagree on how strong a lever freshness
actually is. I'm not claiming the paper is wrong — different portfolio, different metric,
different population — but I'm also not going to quietly drop my own FALSE verdict just
because a published paper says CONFIRMED. Worth flagging in the capstone limitations rather
than picking a side without more data.


In [9]:
# ============================================================
# ML-09 — Section 2
# Rebuild the exact ML-08 modeling dataset
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import HistGradientBoostingRegressor


# ============================================================
# 1. Load the same warehouse tables used by ML-08
# ============================================================

DATA_DIR = "/home/mahad/projects/flyrank-ml-internship-starter/work/data"

fact = pd.read_parquet(
    f"{DATA_DIR}/fact_content_query_90d.parquet"
)

dc = pd.read_parquet(
    f"{DATA_DIR}/dim_content.parquet"
)

print("Loaded fact:", fact.shape)
print("Loaded dim_content:", dc.shape)


# ============================================================
# 2. Recreate ML-08 target: ctr_gap_last30
# ============================================================

fact["ctr_last30"] = (
    fact["clicks_last30"]
    / fact["impressions_last30"].replace(0, np.nan)
)

bins = [0, 3, 5, 10, 20, 50, 1000]
pos_labels = ["1-3", "3-5", "5-10", "10-20", "20-50", "50+"]

fact["pos_bucket_last30"] = pd.cut(
    fact["avg_position_last30"],
    bins=bins,
    labels=pos_labels
).astype(str)

exp_last30 = (
    fact.groupby(
        "pos_bucket_last30",
        observed=False
    )
    .apply(
        lambda g: (
            g["clicks_last30"].sum()
            / g["impressions_last30"].sum()
        ),
        include_groups=False
    )
)

fact["ctr_gap_last30"] = (
    fact["pos_bucket_last30"]
    .map(exp_last30)
    .astype(float)
    - fact["ctr_last30"]
)


# ============================================================
# 3. Exact ML-08 feature definitions
# ============================================================

FACT_FEATURES = [
    "query_char_count",
    "query_token_count",
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_visible_query_count",
    "rare_query_count",
    "rare_impressions_share"
]

DC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "backlinks",
    "char_count",
    "word_count"
]


# ============================================================
# 4. Keep only live content
# ============================================================

dc_live = dc[
    dc["is_published"] & ~dc["is_deleted"]
][
    ["client_hash_id", "content_hash_id"] + DC_FEATURES
].copy()


# ============================================================
# 5. Build the exact ML-08 modeling dataframe
# ============================================================

df = fact[
    [
        "client_hash_id",
        "content_hash_id",
        "query_hash_id",
        "pos_bucket_last30"
    ]
    + FACT_FEATURES
    + ["ctr_gap_last30"]
].merge(
    dc_live,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
).dropna(
    subset=["ctr_gap_last30"]
)


# ============================================================
# 6. Create X, y, and groups
# ============================================================

X = df[FACT_FEATURES + DC_FEATURES].copy()

for col in X.columns:
    if X[col].isnull().any():
        X[col + "_was_missing"] = X[col].isnull().astype(int)
        X[col] = X[col].fillna(X[col].median())

y = df["ctr_gap_last30"].copy()

groups = df["client_hash_id"]


# ============================================================
# 7. Kernel/data sanity check
# ============================================================

print("\nML-09 dataset check")
print("X exists:", "X" in globals())
print("y exists:", "y" in globals())
print("groups exists:", "groups" in globals())
print("X shape:", X.shape)
print("y shape:", y.shape)
print("unique clients:", groups.nunique())
print("feature count:", X.shape[1])


# ============================================================
# 8. Precision@K
# ============================================================

def precision_at_k(scores, y_true, k=50):
    scores = pd.Series(scores).reset_index(drop=True)
    y_true = pd.Series(y_true).reset_index(drop=True)

    top_pred = set(
        scores.sort_values(ascending=False).head(k).index
    )

    top_true = set(
        y_true.sort_values(ascending=False).head(k).index
    )

    return len(top_pred & top_true) / k


# ============================================================
# 9. Run validation under a specified split
# ============================================================

def run_split(splitter, X, y, groups=None):
    scores = []

    if groups is not None:
        splits = splitter.split(X, y, groups=groups)
    else:
        splits = splitter.split(X, y)

    for fold, (tr, te) in enumerate(splits, start=1):

        model = HistGradientBoostingRegressor(
            random_state=42
        )

        model.fit(
            X.iloc[tr],
            y.iloc[tr]
        )

        predictions = model.predict(X.iloc[te])

        score = precision_at_k(
            predictions,
            y.iloc[te],
            k=50
        )

        scores.append(score)

        print(
            f"Fold {fold}: Precision@50 = {score:.4f}"
        )

    return scores


# ============================================================
# 10. Honest grouped-by-client validation
# ============================================================

print("\n" + "=" * 60)
print("GROUPED-BY-CLIENT VALIDATION")
print("=" * 60)

honest_scores = run_split(
    GroupKFold(n_splits=5),
    X,
    y,
    groups=groups
)

honest_mean = np.mean(honest_scores)

print(
    "\nGrouped-by-client Precision@50 per fold:",
    honest_scores
)

print(
    f"Grouped-by-client mean Precision@50: {honest_mean:.4f}"
)


# ============================================================
# 11. Naive random-row validation
# ============================================================

print("\n" + "=" * 60)
print("NAIVE RANDOM-ROW VALIDATION")
print("=" * 60)

naive_scores = run_split(
    KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    X,
    y
)

naive_mean = np.mean(naive_scores)

print(
    "\nRandom-row Precision@50 per fold:",
    naive_scores
)

print(
    f"Random-row mean Precision@50: {naive_mean:.4f}"
)


# ============================================================
# 12. Compare the two validation strategies
# ============================================================

inflation = naive_mean - honest_mean

print("\n" + "=" * 60)
print("VALIDATION COMPARISON")
print("=" * 60)

print(f"Honest grouped mean : {honest_mean:.4f}")
print(f"Naive random mean   : {naive_mean:.4f}")
print(f"Difference          : {inflation:+.4f}")

if inflation > 0:
    print(
        "\nNaive random-row validation produced a higher score."
    )
    print(
        "This shows the potential inflation caused by allowing "
        "the same clients to appear across train and test."
    )
elif inflation < 0:
    print(
        "\nNaive random-row validation produced a lower score "
        "than grouped validation on this run."
    )
else:
    print(
        "\nThe two validation strategies produced the same mean score."
    )

Loaded fact: (2414248, 21)
Loaded dim_content: (519606, 26)

ML-09 dataset check
X exists: True
y exists: True
groups exists: True
X shape: (1814458, 21)
y shape: (1814458,)
unique clients: 49
feature count: 21

GROUPED-BY-CLIENT VALIDATION
Fold 1: Precision@50 = 0.0000
Fold 2: Precision@50 = 0.0000
Fold 3: Precision@50 = 0.0000
Fold 4: Precision@50 = 0.0000
Fold 5: Precision@50 = 0.0000

Grouped-by-client Precision@50 per fold: [0.0, 0.0, 0.0, 0.0, 0.0]
Grouped-by-client mean Precision@50: 0.0000

NAIVE RANDOM-ROW VALIDATION
Fold 1: Precision@50 = 0.0000
Fold 2: Precision@50 = 0.0000
Fold 3: Precision@50 = 0.0000
Fold 4: Precision@50 = 0.0000
Fold 5: Precision@50 = 0.0000

Random-row Precision@50 per fold: [0.0, 0.0, 0.0, 0.0, 0.0]
Random-row mean Precision@50: 0.0000

VALIDATION COMPARISON
Honest grouped mean : 0.0000
Naive random mean   : 0.0000
Difference          : +0.0000

The two validation strategies produced the same mean score.


In [10]:
# No computation needed for this section — it's a direct comparison between the paper's
# reported numbers (already public, in docs/flyrank-seo-research-march-2026.pdf) and this
# project's own ML-06 signal-audit results (already computed and printed there).
print("Finding #3 (CTR vs. position): paper CONFIRMED, my Signal #1: CONFIRMED — agree")
print("Finding #4 (staleness/freshness): paper CONFIRMED, my Signal #2: FALSE — disagree, flagged")


Finding #3 (CTR vs. position): paper CONFIRMED, my Signal #1: CONFIRMED — agree
Finding #4 (staleness/freshness): paper CONFIRMED, my Signal #2: FALSE — disagree, flagged


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**ML-08 already used the grouped-by-client split from the start — there's no "before" version
to compare against, and that's a deliberate choice, not an oversight.** Rather than fabricate a
naive random-split number just to have a before/after table, the honest comparison here is:
run the grouped split (already in ML-08) and ALSO run a naive random split on the same data, so
the *inflation from getting the split wrong* is visible directly, side by side.


In [11]:
# Reuses ML-08's exact X, y, groups (re-run this notebook after ML-08, or paste its cells above
# this one — kept separate here to isolate what changes between splits).
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import HistGradientBoostingRegressor

def precision_at_k(scores, y_true, k=50):
    scores = pd.Series(scores).reset_index(drop=True)
    y_true = pd.Series(y_true).reset_index(drop=True)
    top_pred = set(scores.sort_values(ascending=False).head(k).index)
    top_true = set(y_true.sort_values(ascending=False).head(k).index)
    return len(top_pred & top_true) / k

def run_split(splitter, X, y, groups=None):
    scores = []
    splits = splitter.split(X, y, groups=groups) if groups is not None else splitter.split(X, y)
    for tr, te in splits:
        m = HistGradientBoostingRegressor(random_state=42)
        m.fit(X.iloc[tr], y.iloc[tr])
        scores.append(precision_at_k(m.predict(X.iloc[te]), y.iloc[te]))
    return scores

honest_scores = run_split(GroupKFold(n_splits=5), X, y, groups=groups)
naive_scores = run_split(KFold(n_splits=5, shuffle=True, random_state=42), X, y)

print("Grouped-by-client (honest) Precision@50 per fold:", honest_scores, "mean:", np.mean(honest_scores))
print("Random row split (naive) Precision@50 per fold:", naive_scores, "mean:", np.mean(naive_scores))
print()
print(">>> If naive > honest, that gap IS the inflation from letting the same client leak")
print(">>> across train/test. Report both numbers in the capstone, not just the honest one.")


Grouped-by-client (honest) Precision@50 per fold: [0.0, 0.0, 0.0, 0.0, 0.0] mean: 0.0
Random row split (naive) Precision@50 per fold: [0.0, 0.0, 0.0, 0.0, 0.0] mean: 0.0

>>> If naive > honest, that gap IS the inflation from letting the same client leak
>>> across train/test. Report both numbers in the capstone, not just the honest one.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Same three-part hunt from ML-05, re-run on the exact feature set ML-08's final model used** —
confirming nothing changed between "the contract we wrote" and "the code that actually ran."


In [12]:
LABEL_FIELDS = {"impressions_last30", "clicks_last30", "avg_position_last30"}
EXCLUDED_FIELDS = {"impressions_90d", "clicks_90d", "avg_position_90d", "content_total_impressions_90d",
                    "provider_used", "model_used", "anonymized_impressions_share",
                    "content_updated_date", "last_optimized_date"}

leaked = (LABEL_FIELDS | EXCLUDED_FIELDS) & set(X.columns)
assert not leaked, f"LEAKAGE in final model features: {leaked}"
print("Final feature set clean — no label or excluded-window columns reached the trained model.")

# Re-confirm the "add it back and watch it jump" test still behaves as expected on the final X
X_leaky = X.copy()
X_leaky["impressions_last30"] = df.merge(
    fact[["client_hash_id","content_hash_id","query_hash_id","impressions_last30"]],
    on=["client_hash_id","content_hash_id","query_hash_id"], how="left")["impressions_last30"].values
corr_clean = X.corrwith(y).abs().max()
corr_leaky = X_leaky.corrwith(y).abs().max()
print(f"Max |correlation| with y, clean features: {corr_clean:.4f}")
print(f"Max |correlation| with y, with impressions_last30 added back: {corr_leaky:.4f}")


Final feature set clean — no label or excluded-window columns reached the trained model.
Max |correlation| with y, clean features: 0.0365
Max |correlation| with y, with impressions_last30 added back: 0.0365


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest sentence I've written across this project, and the honest rewrite:**

- Bold (don't say this): *"The model predicts which pages need a refresh."*
- Honest rewrite: *"Using only pre-decision signals, the model produced a ranked list where,
  in cross-validated testing on held-out clients, the top 50 pages overlapped with the top 50
  pages by actual CTR-gap outcome at [X]% precision — compared to [Y]% for a same-information
  fixed-rule baseline. This is decision-support for a human reviewer, not a determination that
  any specific page needs a refresh."*

Fill in `[X]` and `[Y]` from Section 3 of ML-08 once it's run — do not publish this sentence
with placeholder or guessed numbers.


In [13]:
print("Claim rewritten to: observed / measured / directional / decision-support language.")
print("X and Y above must come from ML-08's real Precision@50 output, not a guess.")


Claim rewritten to: observed / measured / directional / decision-support language.
X and Y above must come from ML-08's real Precision@50 output, not a guess.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.